## 07. Modeliranje: Eksperiment C (MC + classical FE + ML)

U ovom notebook-u treniramo klasične ML modele (multi-label klasifikacija) nad Multi-Class (MC) podskupom proteina. Koristimo feature-set kombinacije definisane u notebook-u 03: AAC-CV, TF-IDF + SVD i PH.

### Uvoz biblioteka

In [3]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
import numpy as np
import pandas as pd
import joblib

from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    f1_score, hamming_loss, accuracy_score,
    classification_report, make_scorer
)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

### Učitavanje podataka

Koristimo isti MC train/test split i isti MultiLabelBinarizet koji su definisani u notebook-u 03 (nema ponovnog fitovanja).

In [4]:
DATA_DIR = "../data/processed"
FEATURES_DIR = "../data/features"
MODELS_DIR = "../data/models"
METRICS_DIR = "../results/metrics"
FIGURES_DIR = "../results/figures"

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

RANDOM_STATE = 42

# Entry liste za MC train/test skup (03 notebook)
entries_mc_train = pd.read_csv(os.path.join(FEATURES_DIR, "mc_train_entries.csv"))["Entry"].values
entries_mc_test = pd.read_csv(os.path.join(FEATURES_DIR, "mc_test_entries.csv"))["Entry"].values

# Multi-hot labele (fitovane u 03 - MultiLabelBinarizer)
mlb = joblib.load(os.path.join(FEATURES_DIR, "mc_label_binarizer.pkl"))
label_columns = list(mlb.classes_)

# Labels
mc_labels_df = pd.read_csv(os.path.join(FEATURES_DIR, "mc_labels.csv"), index_col="Entry")
mc_labels_df

y_mc_train = mc_labels_df.loc[entries_mc_train, label_columns].values
y_mc_test = mc_labels_df.loc[entries_mc_test, label_columns].values

print(f"Train skup: {len(entries_mc_train)} proteina")
print(f"Test skup: {len(entries_mc_test)} proteina")
print(f"Klase: {label_columns}")
print(f"y_mc_train: {y_mc_train.shape} | y_mc_test: {y_mc_test.shape}")

Train skup: 5643 proteina
Test skup: 1400 proteina
Klase: ['Hydrolase', 'Receptor', 'Structural protein', 'Transcription factor', 'Transport protein']
y_mc_train: (5643, 5) | y_mc_test: (1400, 5)


### Učitavanje raw obilježja

In [8]:
# AAC-CV je izračunat jednom za cijeli dataset (fiksni vokabular, ne zahtijeva fitovanje)
aac_df = pd.read_csv(os.path.join(FEATURES_DIR, "aac_cv_features.csv"), index_col="Entry")
aac_cols = list(aac_df.columns)

print(f"AAC-CV kolone: {len(aac_cols)}")
aac_df.head()

AAC-CV kolone: 20


,A,C,D,E,F,G,H,I,K,L,M,N,P,Q,R,S,T,V,W,Y
Entry,,,,,,,,,,,,,,,,,,,,
P03986,0.042328,0.031746,0.079365,0.052910,0.037037,0.026455,0.015873,0.074074,0.100529,0.105820,0.015873,0.063492,0.047619,0.031746,0.021164,0.058201,0.095238,0.042328,0.015873,0.042328
Q9NRU3,0.083070,0.021030,0.051525,0.070452,0.037855,0.079916,0.017876,0.028391,0.033649,0.126183,0.012618,0.032597,0.065195,0.030494,0.075710,0.075710,0.057834,0.065195,0.009464,0.025237
Q01167,0.110606,0.006061,0.025758,0.045455,0.021212,0.092424,0.028788,0.046970,0.042424,0.060606,0.012121,0.031818,0.107576,0.057576,0.051515,0.093939,0.068182,0.074242,0.004545,0.018182
P51788,0.102450,0.018931,0.028953,0.062361,0.052339,0.072383,0.017817,0.052339,0.035635,0.110245,0.025612,0.015590,0.060134,0.037862,0.065702,0.073497,0.059020,0.074610,0.015590,0.018931
Q9NXT0,0.042289,0.059701,0.017413,0.097015,0.037313,0.067164,0.074627,0.034826,0.064677,0.069652,0.007463,0.022388,0.032338,0.034826,0.099502,0.114428,0.047264,0.032338,0.004975,0.039801


In [5]:
# Sirove sekvence (fitovanje TF-IDF+SVD dešava se unutar Pipeline-a, po CV foldu)
mc_train_seq_df = pd.read_csv(os.path.join(FEATURES_DIR, "mc_train_sequences_raw.csv"), index_col="Entry")
mc_test_seq_df = pd.read_csv(os.path.join(FEATURES_DIR, "mc_test_sequences_raw.csv"), index_col="Entry")
seq_col = "Sequence"

print(f"Sirove sekvence — train: {mc_train_seq_df.shape[0]} | test: {mc_test_seq_df.shape[0]}")
mc_train_seq_df.head()

Sirove sekvence — train: 5643 | test: 1400


,Sequence
Entry,
P03986,DKQLDADVSPKPTIFLPSIAETKLQKAGTYLCLLEKFFPDIIKIHW...
Q01167,MAAAAAALSGAGTPPAGGGAGGGGAGGGGSPPGGWAVARLEGREFE...
P08574,MAAAAASLRGVVLGPRGAGLPGARARGLLCSARPGQLPLRTPQAVA...
O43678,MAAAAASRGVGAKLGLREIRIHLCQRSPGSQGVRDFIEKRYVELKK...
Q9HD20,MAAAAAVGNAVPCGARPCGVRPDGQPKPGPQPRALLAAGPALIANG...


In [6]:
# Sirove (neskalirane) physchem osobine (StandardScaler fituje se unutar Pipeline-a, po CV foldu)
mc_train_ph_df = pd.read_csv(os.path.join(FEATURES_DIR, "mc_train_physchem_raw.csv"), index_col="Entry")
mc_test_ph_df = pd.read_csv(os.path.join(FEATURES_DIR, "mc_test_physchem_raw.csv"), index_col="Entry")
ph_cols = list(mc_train_ph_df.columns)

print(f"PH kolone: {ph_cols}")
mc_train_ph_df.head()

PH kolone: ['MW', 'pI', 'GRAVY', 'Aromaticity', 'Instability']


,MW,pI,GRAVY,Aromaticity,Instability
Entry,,,,,
P03986,21697.6278,5.954646,-0.401058,0.095238,29.608995
Q01167,69061.1230,9.564739,-0.361061,0.043939,57.529864
P08574,35421.5966,9.150336,-0.137538,0.083077,37.608308
O43678,10921.4343,9.619666,-0.311111,0.060606,38.597980
Q9HD20,132953.2845,8.464006,0.114037,0.083056,43.975997


### Sastavljanje DataFrame-a 

Spajamo AAC-CV, raw sekvence i neskalirane PH vrijednosti u jedan DataFrame po Entry-ju, poravnat sa redoslijedom labela (y_sc_train/y_sc_test).

In [9]:
X_train_raw = aac_df.join(mc_train_seq_df, how="inner").join(mc_train_ph_df, how="inner") 
X_test_raw = aac_df.join(mc_test_seq_df, how="inner").join(mc_test_ph_df, how="inner") 

# Poravnanje redoslijeda redova sa redoslijedom labela
X_train_raw = X_train_raw.loc[entries_mc_train]
X_test_raw = X_test_raw.loc[entries_mc_test]

assert list(X_train_raw.index) == list(entries_mc_train)
assert list(X_test_raw.index) == list(entries_mc_test)

print(f"X_train_raw: {X_train_raw.shape} | X_test_raw: {X_test_raw.shape}")
X_train_raw.head()

X_train_raw: (5643, 26) | X_test_raw: (1400, 26)


,A,C,D,E,F,G,H,I,K,L,...,T,V,W,Y,Sequence,MW,pI,GRAVY,Aromaticity,Instability
Entry,,,,,,,,,,,,,,,,,,,,,
P03986,0.042328,0.031746,0.079365,0.052910,0.037037,0.026455,0.015873,0.074074,0.100529,0.105820,...,0.095238,0.042328,0.015873,0.042328,DKQLDADVSPKPTIFLPSIAETKLQKAGTYLCLLEKFFPDIIKIHW...,21697.6278,5.954646,-0.401058,0.095238,29.608995
Q01167,0.110606,0.006061,0.025758,0.045455,0.021212,0.092424,0.028788,0.046970,0.042424,0.060606,...,0.068182,0.074242,0.004545,0.018182,MAAAAAALSGAGTPPAGGGAGGGGAGGGGSPPGGWAVARLEGREFE...,69061.1230,9.564739,-0.361061,0.043939,57.529864
P08574,0.116923,0.018462,0.043077,0.046154,0.030769,0.086154,0.030769,0.018462,0.043077,0.113846,...,0.030769,0.061538,0.009231,0.043077,MAAAAASLRGVVLGPRGAGLPGARARGLLCSARPGQLPLRTPQAVA...,35421.5966,9.150336,-0.137538,0.083077,37.608308
O43678,0.121212,0.020202,0.040404,0.060606,0.030303,0.070707,0.010101,0.050505,0.060606,0.101010,...,0.020202,0.070707,0.010101,0.020202,MAAAAASRGVGAKLGLREIRIHLCQRSPGSQGVRDFIEKRYVELKK...,10921.4343,9.619666,-0.311111,0.060606,38.597980
Q9HD20,0.091362,0.024086,0.039037,0.055648,0.047342,0.058140,0.022425,0.047342,0.050664,0.127076,...,0.051495,0.082226,0.010797,0.024917,MAAAAAVGNAVPCGARPCGVRPDGQPKPGPQPRALLAAGPALIANG...,132953.2845,8.464006,0.114037,0.083056,43.975997


### Definisanje feature-ser kombinacija (kroz ColumnTransfer)

In [10]:
# Funkcija koja bira odgovarajuci transformator i vraca Pipeline
# (svaka feature-set/model kombinacija ima sopstveni transformator)
def make_tfidf_svd_step():
    return Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(2, 3), max_features=3000, lowercase=False)),
        ("svd", TruncatedSVD(n_components=150, random_state=RANDOM_STATE)),
    ])

# Funkcija koja vraca ColumnTransfer za odgovarajucu feature-set kombinaciju
def make_preprocessor(feature_set_name):
    if feature_set_name == "AAC-CV":
        transformers = [("aac", "passthrough", aac_cols)]
    elif feature_set_name == "TFIDF-SVD":
        transformers = [("tfidf-svd", make_tfidf_svd_step(), seq_col)]
    elif feature_set_name == "PH":
        transformers = [("ph_scaled", StandardScaler(), ph_cols)]
    elif feature_set_name == "AAC-CV+TFIDF-SVD":
        transformers = [
            ("aac", "passthrough", aac_cols),
            ("tfidf-svd", make_tfidf_svd_step(), seq_col),
        ]
    elif feature_set_name == "AAC-CV+PH":
        transformers = [
            ("aac", "passthrough", aac_cols),
            ("ph_scaled", StandardScaler(), ph_cols),
        ]
    elif feature_set_name == "TFIDF-SVD+PH":
        transformers = [
            ("tfidf-svd", make_tfidf_svd_step(), seq_col),
            ("ph_scaled", StandardScaler(), ph_cols),
        ]
    elif feature_set_name == "AAC-CV+TFIDF-SVD+PH":
        transformers = [
            ("aac", "passthrough", aac_cols),
            ("tfidf-svd", make_tfidf_svd_step(), seq_col),
            ("ph_scaled", StandardScaler(), ph_cols),
        ]
    else:
        raise ValueError(f"Nepoznat feature set: {feature_set_name}")

    return ColumnTransformer(transformers=transformers, remainder="drop")

FEATURE_SET_NAMES = [
    "AAC-CV",
    "TFIDF-SVD",
    "PH",
    "AAC-CV+TFIDF-SVD",
    "AAC-CV+PH",
    "TFIDF-SVD+PH",
    "AAC-CV+TFIDF-SVD+PH",
]


### AAC-CV

In [ ]:
"""# AAC-CV je izračunat jednom za cijeli dataset (fiksni vokabular, ne zahtijeva fitovanje)
# Selektujemo redove koji pripadaju MC train/test skupu
aac_df = pd.read_csv(os.path.join(FEATURES_DIR, "aac_cv_features.csv"), index_col="Entry")

mc_train_aac = aac_df.loc[entries_mc_train].values
mc_test_aac = aac_df.loc[entries_mc_test].values

print(f"AAC-CV train: {mc_train_aac.shape} | test: {mc_test_aac.shape}")"""

AAC-CV train: (5643, 20) | test: (1400, 20)


### TF-IDF + SVD

In [ ]:
"""# Fitovano na MC train skupu u 03
mc_train_tfidf = np.load(os.path.join(FEATURES_DIR, "mc_train_tfidf_svd.npy")) 
mc_test_tfidf = np.load(os.path.join(FEATURES_DIR, "mc_test_tfidf_svd.npy")) 

print(f"TF-IDF-SVD train: {mc_train_tfidf.shape} | test: {mc_test_tfidf.shape}")"""

TF-IDF-SVD train: (5643, 150) | test: (1400, 150)


### Fizičko-hemijske osobine

In [ ]:
"""mc_train_ph = np.load(os.path.join(FEATURES_DIR, "mc_train_physchem_scaled.npy"))
mc_test_ph = np.load(os.path.join(FEATURES_DIR, "mc_test_physchem_scaled.npy"))

print(f"PH train: {mc_train_ph.shape} | test: {mc_test_ph.shape}")"""

PH train: (5643, 5) | test: (1400, 5)


### Kombinovanje u feature setove

In [ ]:
"""feature_sets_train = {
    "AAC-CV": mc_train_aac,
    "TFIDF-SVD": mc_train_tfidf,
    "PH": mc_train_ph,
    "AAC-CV+TFIDF-SVD": np.hstack([mc_train_aac, mc_train_tfidf]),
    "AAC-CV+PH": np.hstack([mc_train_aac, mc_train_ph]),
    "TFIDF-SVD+PH": np.hstack([mc_train_tfidf, mc_train_ph]),
    "AAC-CV+TFIDF-SVD+PH": np.hstack([mc_train_aac, mc_train_tfidf, mc_train_ph]),
}

feature_sets_test = {
    "AAC-CV": mc_test_aac,
    "TFIDF-SVD": mc_test_tfidf,
    "PH": mc_test_ph,
    "AAC-CV+TFIDF-SVD": np.hstack([mc_test_aac, mc_test_tfidf]),
    "AAC-CV+PH": np.hstack([mc_test_aac, mc_test_ph]),
    "TFIDF-SVD+PH": np.hstack([mc_test_tfidf, mc_test_ph]),
    "AAC-CV+TFIDF-SVD+PH": np.hstack([mc_test_aac, mc_test_tfidf, mc_test_ph]),
}

for name, matrix in feature_sets_train.items():
    print(f"{name:<20} train: {matrix.shape} test: {feature_sets_test[name].shape}")"""

AAC-CV               train: (5643, 20) test: (1400, 20)
TFIDF-SVD            train: (5643, 150) test: (1400, 150)
PH                   train: (5643, 5) test: (1400, 5)
AAC-CV+TFIDF-SVD     train: (5643, 170) test: (1400, 170)
AAC-CV+PH            train: (5643, 25) test: (1400, 25)
TFIDF-SVD+PH         train: (5643, 155) test: (1400, 155)
AAC-CV+TFIDF-SVD+PH  train: (5643, 175) test: (1400, 175)


### Definisanje modela i mreže hiperparametara

In [14]:
param_grids = {
    "LogisticRegression": {
        "model": OneVsRestClassifier(
            LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)
        ),
        "params": {
            "estimator__C": [0.01, 0.1, 1, 10, 100],
        },
    },
    "RandomForest": {
        "model": OneVsRestClassifier(
            RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE)
        ),
        "params": {
            "estimator__n_estimators": [200],
            "estimator__max_depth": [10, 25],
            "estimator__min_samples_leaf": [2, 5],
        },
    },
    "SVM": {
        "model": OneVsRestClassifier(
            SVC(class_weight="balanced", probability=True, random_state=RANDOM_STATE)
        ),
        "params": {
            "estimator__C": [0.1, 1, 10],
            "estimator__kernel": ["rbf", "linear"],
        },
    },
}

f1_micro_scorer = make_scorer(f1_score, average="micro", zero_division=0)
cv = MultilabelStratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

### Glavna petlja treniranja

In [17]:
from joblib import Memory

cache_dir = "../cache/pipeline_cache"
memory = Memory(location=cache_dir, verbose=0)

all_results = []

for feature_name in FEATURE_SET_NAMES:
    preprocessor = make_preprocessor(feature_name)

    for model_name, config in param_grids.items():
        print(f"Treniranje: {model_name:<20} | Feature set: {feature_name}")

        full_pipeline = Pipeline([
            ("features", preprocessor),
            ("model", config["model"]),
        ], memory=memory)

        pipeline_param_grid = {
            f"model__{param_name}": values 
            for param_name, values in config["params"].items()
        }

        grid = GridSearchCV(
            estimator=full_pipeline,
            param_grid=pipeline_param_grid,
            cv=cv,
            scoring=f1_micro_scorer,
            n_jobs=-1,
            refit=True,
        )

        grid.fit(X_train_raw, y_mc_train)

        best_pipeline = grid.best_estimator_

        # Predikcije na train skupu (isti fit-ovan model, bez dodatnog treniranja)
        y_train_pred = best_pipeline.predict(X_train_raw)
        train_f1_micro = f1_score(y_mc_train, y_train_pred, average="micro", zero_division=0)
        train_f1_macro = f1_score(y_mc_train, y_train_pred, average="macro", zero_division=0)
        train_acc = accuracy_score(y_mc_train, y_train_pred) # subser accuracy
                
        # Predikcije na test skupu
        y_test_pred = best_pipeline.predict(X_test_raw)
        test_f1_micro = f1_score(y_mc_test, y_test_pred, average="micro", zero_division=0)
        test_f1_macro = f1_score(y_mc_test, y_test_pred, average="macro", zero_division=0)
        test_f1_samples = f1_score(y_mc_test, y_test_pred, average="samples", zero_division=0)
        test_acc = accuracy_score(y_mc_test, y_test_pred)  # subset accuracy (exact match)
        test_hamming = hamming_loss(y_mc_test, y_test_pred)
        
        best_params = {
            k.replace("model__", ""): v for k, v in grid.best_params_.items()
        }

        all_results.append({
            "Feature set": feature_name,
            "Model": model_name,
            "Best params": grid.best_params_,

            "CV F1 micro": grid.best_score_,

            "Train F1 micro": train_f1_micro,
            "Train F1 macro": train_f1_macro,
            "Train Subset Accuracy": train_acc,

            "Test F1 micro": test_f1_micro,
            "Test F1 macro": test_f1_macro,
            "Test F1 samples": test_f1_samples,
            "Test Subset Accuracy": test_acc,
            "Test Hamming Loss": test_hamming,
            
            "Overfit Gap (F1 micro)": train_f1_micro - test_f1_micro,
        })

        model_filename = f"mc_{model_name}_{feature_name}.pkl"
        joblib.dump(best_pipeline, os.path.join(MODELS_DIR, model_filename))

    memory.clear()

print("\nTreniranje završeno.")

Treniranje: LogisticRegression   | Feature set: AAC-CV


Treniranje: RandomForest         | Feature set: AAC-CV
Treniranje: SVM                  | Feature set: AAC-CV


[Memory(location=../cache/pipeline_cache\joblib)]: Flushing completely the cache


Treniranje: LogisticRegression   | Feature set: TFIDF-SVD
Treniranje: RandomForest         | Feature set: TFIDF-SVD
Treniranje: SVM                  | Feature set: TFIDF-SVD


[Memory(location=../cache/pipeline_cache\joblib)]: Flushing completely the cache


...

Treniranje: LogisticRegression   | Feature set: TFIDF-SVD+PH
Treniranje: RandomForest         | Feature set: TFIDF-SVD+PH
Treniranje: SVM                  | Feature set: TFIDF-SVD+PH


[Memory(location=../cache/pipeline_cache\joblib)]: Flushing completely the cache


Treniranje: LogisticRegression   | Feature set: AAC-CV+TFIDF-SVD+PH
Treniranje: RandomForest         | Feature set: AAC-CV+TFIDF-SVD+PH
Treniranje: SVM                  | Feature set: AAC-CV+TFIDF-SVD+PH


[Memory(location=../cache/pipeline_cache\joblib)]: Flushing completely the cache



Treniranje završeno.


In [ ]:
"""all_results = []

for feature_name, X_train in feature_sets_train.items():
    X_test = feature_sets_test[feature_name]

    for model_name, config in param_grids.items():
        print(f"Treniranje: {model_name:<20} | Feature set: {feature_name}")

        grid = GridSearchCV(
            estimator=config["model"],
            param_grid=config["params"],
            cv=cv,
            scoring=f1_micro_scorer,
            n_jobs=-1,
            refit=True,
        )
        grid.fit(X_train, y_mc_train)

        best_model = grid.best_estimator_

        # Predikcije na train skupu (isti fit-ovan model, bez dodatnog treniranja)
        y_train_pred = best_model.predict(X_train)
        train_f1_micro = f1_score(y_mc_train, y_train_pred, average="micro", zero_division=0)
        train_f1_macro = f1_score(y_mc_train, y_train_pred, average="macro", zero_division=0)
        train_acc = accuracy_score(y_mc_train, y_train_pred)  # subset accuracy (exact match)

        # Predikcije na test skupu
        y_test_pred = best_model.predict(X_test)
        test_f1_micro = f1_score(y_mc_test, y_test_pred, average="micro", zero_division=0)
        test_f1_macro = f1_score(y_mc_test, y_test_pred, average="macro", zero_division=0)
        test_f1_samples = f1_score(y_mc_test, y_test_pred, average="samples", zero_division=0)
        test_acc = accuracy_score(y_mc_test, y_test_pred)  # subset accuracy (exact match)
        test_hamming = hamming_loss(y_mc_test, y_test_pred)

        all_results.append({
            "Feature set": feature_name,
            "Model": model_name,
            "Best params": grid.best_params_,

            "CV F1 micro": grid.best_score_,

            "Train F1 micro": train_f1_micro,
            "Train F1 macro": train_f1_macro,
            "Train Subset Accuracy": train_acc,

            "Test F1 micro": test_f1_micro,
            "Test F1 macro": test_f1_macro,
            "Test F1 samples": test_f1_samples,
            "Test Subset Accuracy": test_acc,
            "Test Hamming Loss": test_hamming,
            
            "Overfit Gap (F1 micro)": train_f1_micro - test_f1_micro,
        })

        model_filename = f"mc_{model_name}_{feature_name}.pkl"
        joblib.dump(best_model, os.path.join(MODELS_DIR, model_filename))

print("\nTreniranje završeno.")"""

Treniranje: LogisticRegression   | Feature set: AAC-CV


Treniranje: RandomForest         | Feature set: AAC-CV
Treniranje: SVM                  | Feature set: AAC-CV
Treniranje: LogisticRegression   | Feature set: TFIDF-SVD
Treniranje: RandomForest         | Feature set: TFIDF-SVD
Treniranje: SVM                  | Feature set: TFIDF-SVD
Treniranje: LogisticRegression   | Feature set: PH
Treniranje: RandomForest         | Feature set: PH
Treniranje: SVM                  | Feature set: PH
Treniranje: LogisticRegression   | Feature set: AAC-CV+TFIDF-SVD
Treniranje: RandomForest         | Feature set: AAC-CV+TFIDF-SVD
Treniranje: SVM                  | Feature set: AAC-CV+TFIDF-SVD
Treniranje: LogisticRegression   | Feature set: AAC-CV+PH
Treniranje: RandomForest         | Feature set: AAC-CV+PH
Treniranje: SVM                  | Feature set: AAC-CV+PH
Treniranje: LogisticRegression   | Feature set: TFIDF-SVD+PH
Treniranje: RandomForest         | Feature set: TFIDF-SVD+PH
Treniranje: SVM                  | Feature set: TFIDF-SVD+PH
Treniranje:

c:\Users\Korisnik\AppData\Local\Programs\Python\Python310\lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Treniranje završeno.


### Pregled i čuvanje rezultata

In [18]:
results_df = pd.DataFrame(all_results)

# Sortiranje po Test F1 micro
results_df = results_df.sort_values(by="CV F1 micro", ascending=False).reset_index(drop=True)
results_df.to_csv(os.path.join(METRICS_DIR, "mc_classical_modeling_results.csv"), index=False)

display_cols = [
    "Model", "Feature set",
    "Train F1 micro", "Train Subset Accuracy",
    "CV F1 micro",
    "Test F1 micro", "Test F1 macro", "Test F1 samples",
    "Test Subset Accuracy", "Test Hamming Loss",
    "Overfit Gap (F1 micro)",
]

results_df[display_cols].style.format({
    "Train F1 micro": "{:.3f}",
    "Train Subset Accuracy": "{:.3f}",
    "CV F1 micro": "{:.3f}",
    "Test F1 micro": "{:.3f}",
    "Test F1 macro": "{:.3f}",
    "Test F1 samples": "{:.3f}",
    "Test Subset Accuracy": "{:.3f}",
    "Test Hamming Loss": "{:.3f}",
    "Overfit Gap (F1 micro)": "{:.3f}",
}).background_gradient(subset=["Overfit Gap (F1 micro)"], cmap="Reds")

,Model,Feature set,Train F1 micro,Train Subset Accuracy,CV F1 micro,Test F1 micro,Test F1 macro,Test F1 samples,Test Subset Accuracy,Test Hamming Loss,Overfit Gap (F1 micro)
0,SVM,AAC-CV+TFIDF-SVD,0.985,0.968,0.797,0.816,0.806,0.804,0.729,0.079,0.169
1,SVM,TFIDF-SVD,0.989,0.977,0.794,0.814,0.802,0.799,0.729,0.079,0.174
2,SVM,TFIDF-SVD+PH,0.882,0.774,0.767,0.770,0.756,0.780,0.651,0.105,0.111
3,SVM,AAC-CV+TFIDF-SVD+PH,0.881,0.774,0.767,0.770,0.756,0.780,0.651,0.105,0.111
4,RandomForest,AAC-CV+PH,0.961,0.927,0.740,0.753,0.741,0.719,0.667,0.100,0.208
5,RandomForest,AAC-CV+TFIDF-SVD+PH,0.921,0.844,0.733,0.742,0.725,0.717,0.634,0.108,0.179
6,RandomForest,TFIDF-SVD+PH,0.916,0.837,0.728,0.726,0.704,0.701,0.621,0.115,0.190
7,RandomForest,AAC-CV+TFIDF-SVD,0.927,0.858,0.723,0.738,0.716,0.703,0.633,0.107,0.188
8,SVM,AAC-CV,0.809,0.649,0.720,0.732,0.721,0.754,0.569,0.129,0.077
9,RandomForest,AAC-CV,0.883,0.775,0.717,0.729,0.717,0.712,0.611,0.117,0.154
